# 貴人合盤分析器 — 互動教學

本 Notebook 教你如何使用 `guiren-analyzer` skill，從姓名學五格判定兩人之間的「貴人 / 小人 / 平宮」關係。

---

## 目錄

1. [基礎概念：什麼是貴人 / 小人 / 平宮？](#ch1)
2. [核心算法：化氣 reduce](#ch2)
3. [五格速查表](#ch3)
4. [實作：用 Python 跑分析](#ch4)
5. [進階：護貴人 / 合盤上吉上凶](#ch5)
6. [練習題](#ch6)

<a id='ch1'></a>
## 1. 基礎概念

姓名學的「合盤」是用兩人的五格（天/人/地/外/總）做交叉比對。核心有三種關係：

| 關係 | 比對方式 | 意義 |
|------|----------|------|
| **貴人** | A 的人格 vs B 的地格 | A 會幫 B（拉拔基礎） |
| **小人** | A 的總格 vs B 的地格 | A 會拖累 B（結果衝擊基礎） |
| **平宮** | 同格相等（人/地/總） | 兩人思維或目標相近 |

> **重要**：天格和外格**不參與**貴人/小人/平宮判定。

### 直接 vs 間接

| 類型 | 條件 | 力量 |
|------|------|------|
| 直接貴人 | A.人格 **實數** = B.地格 | 1x（直接給資源） |
| 間接貴人 | A.人格 **化氣** = B.地格 **化氣** | **10x**（介紹資源，力量更強） |
| 明小人 | A.總格 **實數** = B.地格 | 直接衝突 |
| 暗小人 | A.總格 **化氣** = B.地格 **化氣** | 無心之過 |

> 課程原話：「間接貴人的力量是直接貴人的**十倍**」
>
> 直接和間接**獨立判定**，可以同時成立。

<a id='ch2'></a>
## 2. 核心算法：化氣 reduce

化氣就是把筆劃數字反覆拆位相加，直到變成個位數（1-9）。

In [ ]:
def reduce(n):
    """姓名學化氣：反覆拆位相加直到個位數"""
    n = abs(int(n))
    while n > 9:
        n = sum(int(c) for c in str(n))
    return n

# 試試看
examples = [10, 15, 24, 25, 32, 33, 99]
for x in examples:
    print(f'  {x:3d} --> 化氣 {reduce(x)}')

### 動手試

修改下面的數字，看看化氣結果：

In [ ]:
# 在這裡輸入任意筆劃
my_stroke = 47

print(f'{my_stroke} 的化氣 = {reduce(my_stroke)}')
print(f'拆解過程：', end='')
n = my_stroke
steps = [str(n)]
while n > 9:
    n = sum(int(c) for c in str(n))
    steps.append(str(n))
print(' -> '.join(steps))

<a id='ch3'></a>
## 3. 五格速查表

三字單姓的五格公式：

```
姓 = s1, 名一 = s2, 名二 = s3

天格 = s1 + 1
人格 = s1 + s2
地格 = s2 + s3
外格 = s3 + 1
總格 = s1 + s2 + s3
```

### 示範：李佳穎（康熙筆劃 8 + 8 + 16）

In [ ]:
def calc_grids_3char(s1, s2, s3):
    """三字單姓的五格計算"""
    return {
        '天格': s1 + 1,
        '人格': s1 + s2,
        '地格': s2 + s3,
        '外格': s3 + 1,
        '總格': s1 + s2 + s3,
    }

li = calc_grids_3char(8, 8, 16)  # 李佳穎
print('李佳穎 五格：')
for k, v in li.items():
    print(f'  {k} = {v}  (化氣 {reduce(v)})')

<a id='ch4'></a>
## 4. 實作：用 Python 跑貴人分析

### 方法一：直接呼叫 guiren.py CLI

In [ ]:
import subprocess, os, pathlib

# 自動偵測 repo root（從 notebook 所在的 docs/ 往上找）
repo_root = pathlib.Path(os.getcwd())
if repo_root.name == 'docs':
    repo_root = repo_root.parent

# 找到 guiren.py 的路徑（repo 內 > user-level > root-level）
candidates = [
    repo_root / '.claude' / 'skills' / 'guiren-analyzer' / 'guiren.py',
    pathlib.Path.home() / '.claude' / 'skills' / 'guiren-analyzer' / 'guiren.py',
    pathlib.Path('/root/.claude/skills/guiren-analyzer/guiren.py'),
]
guiren_path = next((str(p) for p in candidates if p.exists()), None)
assert guiren_path, f'找不到 guiren.py！候選路徑：{[str(p) for p in candidates]}'
print(f'guiren.py 路徑: {guiren_path}')

In [ ]:
# 跑一個範例：李佳穎 vs 王大明
result = subprocess.run(
    ['python3', guiren_path,
     '--a', '李佳穎:9,16,24,17,32',
     '--b', '王大明:11,15,24,9,33'],
    capture_output=True, text=True
)
print(result.stdout)

### 方法二：import 模組直接操作

In [ ]:
import sys, importlib

# 把 skill 目錄加到 Python path
skill_dir = os.path.dirname(guiren_path)
if skill_dir not in sys.path:
    sys.path.insert(0, skill_dir)

import guiren
importlib.reload(guiren)  # 確保載入最新版

# 建立兩個人的資料
a = guiren.Person('李佳穎', tian=9, ren=16, di=24, wai=17, zong=32)
b = guiren.Person('王大明', tian=11, ren=15, di=24, wai=9, zong=33)

# 跑分析
report = guiren.analyze(a, b)

# 文字報告
print(guiren.render_text(report))

### 圖解：方向性很重要！

```
        A.人格 ─────→ B.地格    =  A 是 B 的「貴人」
        B.人格 ─────→ A.地格    =  B 是 A 的「貴人」
        A.總格 ─────→ B.地格    =  A 是 B 的「小人」
        B.總格 ─────→ A.地格    =  B 是 A 的「小人」
```

A→B 和 B→A 是**兩條獨立的線**，每條都要檢查。

### 拆解看判定過程

In [ ]:
print('=' * 60)
print(f'{a.name} vs {b.name} — 逐線判定')
print('=' * 60)

checks = [
    ('貴人 (A→B)', '人格→地格', a.ren, b.di, '直接貴人', '間接貴人'),
    ('貴人 (B→A)', '人格→地格', b.ren, a.di, '直接貴人', '間接貴人'),
    ('小人 (A→B)', '總格→地格', a.zong, b.di, '明小人', '暗小人'),
    ('小人 (B→A)', '總格→地格', b.zong, a.di, '暗小人', '暗小人'),
]

for title, pair, val_a, val_b, direct_label, indirect_label in checks:
    ha, hb = guiren.reduce_digit(val_a), guiren.reduce_digit(val_b)
    direct = val_a == val_b
    indirect = ha == hb
    
    status = []
    if direct:   status.append(f'  {direct_label} ({val_a} == {val_b})')
    if indirect: status.append(f'  {indirect_label} (化氣 {ha} == {hb})')
    if not status: status.append('  (無)')
    
    print(f'\n【{title}】 {pair}: {val_a} vs {val_b}')
    for s in status:
        print(s)

### 查看旗標與結論

In [ ]:
print('旗標 (flags):')
flag_labels = {
    'mutual_noble':   '雙向互為貴人（合盤上吉）',
    'mutual_villain': '雙向互為小人（合盤上凶）',
    'huguiren':       '護貴人（雙向間接貴人，超強緣分）',
    'a_to_b_noble':   f'{a.name}→{b.name} 貴人',
    'b_to_a_noble':   f'{b.name}→{a.name} 貴人',
    'a_to_b_villain': f'{a.name}→{b.name} 小人',
    'b_to_a_villain': f'{b.name}→{a.name} 小人',
}
for key, label in flag_labels.items():
    val = report.flags[key]
    icon = '  ' if not val else '  '
    print(f'  {icon} {label}: {val}')

print(f'\n五行俱全: {a.name}={report.a_wuxing_complete}, {b.name}={report.b_wuxing_complete}')
print(f'\n結論: {report.verdict}')

<a id='ch5'></a>
## 5. 進階：護貴人 / 合盤上吉上凶

| 旗標 | 條件 | 解讀 |
|------|------|------|
| 護貴人 | A→B **間接**貴人 + B→A **間接**貴人 | 課程「兩條就超強」 |
| 合盤上吉 | A→B 貴人（任意）+ B→A 貴人（任意） | 雙方都受益 |
| 合盤上凶 | A→B 小人 + B→A 小人 | 雙方互相拖累 |
| 亦正亦邪 | 同方向同時有貴人 + 小人 | 幫你的人也拖你 |

### 實測「護貴人」案例

In [ ]:
# 構造一個護貴人的例子：
# A.人=15 化氣6, B.地=24 化氣6 → A→B 間接貴人
# B.人=14 化氣5, A.地=23 化氣5 → B→A 間接貴人
# 雙向都是間接貴人 → 護貴人！

hu_a = guiren.Person('甲某', tian=10, ren=15, di=23, wai=10, zong=30)
hu_b = guiren.Person('乙某', tian=10, ren=14, di=24, wai=10, zong=30)

hu_report = guiren.analyze(hu_a, hu_b)
print(guiren.render_text(hu_report))

### 實測「合盤上凶」案例

In [ ]:
# 構造一個合盤上凶的例子：
# A.總=24 = B.地=24 → A→B 明小人
# B.總=20 = A.地=20 → B→A 明小人
# 兩邊都是小人 → 合盤上凶！

bad_a = guiren.Person('壞甲', tian=10, ren=11, di=20, wai=10, zong=24)
bad_b = guiren.Person('壞乙', tian=10, ren=11, di=24, wai=10, zong=20)

bad_report = guiren.analyze(bad_a, bad_b)
print(guiren.render_text(bad_report))

### 實測「亦正亦邪」案例

In [ ]:
# 同方向同時觸發「貴人 + 小人」
# B.人=15 化氣6 = A.地=24 化氣6 → B→A 間接貴人
# B.總=33 化氣6 = A.地=24 化氣6 → B→A 暗小人

mix_a = guiren.Person('李佳穎', tian=9, ren=16, di=24, wai=17, zong=32)
mix_b = guiren.Person('王大明', tian=11, ren=15, di=24, wai=9, zong=33)

mix_report = guiren.analyze(mix_a, mix_b)
print(f'\n結論: {mix_report.verdict}')
print('\n注意 verdict 會列出具體標籤（間接貴人、暗小人），不只說「貴人也是小人」')

<a id='ch6'></a>
## 6. 練習題

### 練習 1：手動判定

A 的五格：天 10 / 人 24 / 地 20 / 外 11 / 總 34

B 的五格：天 11 / 人 20 / 地 24 / 外 10 / 總 30

**問題**：
1. A 是 B 的貴人嗎？（直接 or 間接？）
2. B 是 A 的貴人嗎？
3. 有沒有小人線？
4. 有沒有平宮？
5. 會觸發什麼旗標？

先用紙筆算，再跑下面的 cell 對答案：

In [ ]:
# 練習 1 解答（先自己算再執行！）
q1_a = guiren.Person('A', tian=10, ren=24, di=20, wai=11, zong=34)
q1_b = guiren.Person('B', tian=11, ren=20, di=24, wai=10, zong=30)
q1_report = guiren.analyze(q1_a, q1_b)
print(guiren.render_text(q1_report))

### 練習 2：換你的名字試試看

把下面的五格換成你自己 + 你想測的人：

In [ ]:
# 換成你自己的五格（把 0 改成實際筆劃數再執行）
my = guiren.Person(
    '你的名字',
    tian=9,    # <- 填入你的天格
    ren=16,    # <- 填入你的人格
    di=24,     # <- 填入你的地格
    wai=17,    # <- 填入你的外格
    zong=32,   # <- 填入你的總格
)

target = guiren.Person(
    '對方名字',
    tian=11,   # <- 填入對方天格
    ren=15,    # <- 填入對方人格
    di=24,     # <- 填入對方地格
    wai=9,     # <- 填入對方外格
    zong=33,   # <- 填入對方總格
)

my_report = guiren.analyze(my, target)
print(guiren.render_text(my_report))

### 練習 3：JSON 輸出（給程式用）

In [ ]:
import json

# 用 JSON 格式看報告（可以餵給其他程式）
report_dict = guiren.report_to_dict(report)
print(json.dumps(report_dict, ensure_ascii=False, indent=2))

---

## 快速參考

### CLI 指令

```bash
# 基本分析
python3 .claude/skills/guiren-analyzer/guiren.py \
    --a "名字A:天,人,地,外,總" --b "名字B:天,人,地,外,總"

# JSON 輸出
python3 .claude/skills/guiren-analyzer/guiren.py --a ... --b ... --json

# 自驗（跑內建斷言）
python3 .claude/skills/guiren-analyzer/guiren.py --assert
```

### Claude Code 自然語言觸發

只要跟 Claude 說以下這些話，skill 會自動觸發：

- 「某某是不是我的貴人？」
- 「我和 XX 合盤如何？」
- 「老闆會不會拖累我？」
- 「面試官是貴人嗎？」
- 「兩個人的緣分」

### 判定規則一覽

```
貴人：A.人格 → B.地格（直接=實數相同，間接=化氣相同）
小人：A.總格 → B.地格（明=實數相同，暗=化氣相同）
平宮：只看 人/地/總 三格（天/外不看）
護貴人：雙向都有間接貴人線
合盤上吉：雙向互為貴人
合盤上凶：雙向互為小人
```